*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 14: Capstone Project 2: Advanced Generative AI with Lightning. It refactors the denoising autoencoder into a Lightning app and extends the same foundation toward a variational autoencoder.

This capstone is meant to feel like an integration exercise: the same core ideas from the book are reorganized into modular, reusable components that can be tested, trained, and extended.

## Capstone 2: Advanced Generative AI with Lightning

## Project A: Refactoring the Denoising Autoencoder
### Configuring the DataModule: From Step 1 to 4

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl

# The DataModule provides a reusable interface that separates data concerns from model logic.
# Lightning manages the lifecycle: prepare_data() for downloads, setup() for train/val/test splits,
# and loader methods that Trainer calls at the appropriate time.
class CIFARDataModule(pl.LightningDataModule):
    # Step 1: Configure the DataModule with paths, batch size, and data loading parameters.
    # These settings are stored as hyperparameters for easy inspection and logging.
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 256,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.transform = transforms.ToTensor()
        self.num_workers = min(4, os.cpu_count() or 1)
        self.pin_memory = torch.cuda.is_available()

    # Step 2: Download Without Creating Process-Local State
    # prepare_data() runs only once per cluster and does NOT create instance attributes.
    # Lightning designates one process to download and others wait, avoiding race conditions.
    def prepare_data(self):
        datasets.CIFAR100(
            self.hparams.data_dir,
            train=True,
            download=True,
        )
        datasets.CIFAR100(
            self.hparams.data_dir,
            train=False,
            download=True,
        )

    # Step 3: Construct Deterministic Stage State
    # setup() runs in every participating process and creates train/val/test splits.
    # Using a fixed random seed (42) ensures all processes split data identically.
    def setup(self, stage: str | None = None):
        if stage in ("fit", "validate", None):
            if not hasattr(self, "train_set"):
                full_dataset = datasets.CIFAR100(
                    self.hparams.data_dir,
                    train=True,
                    transform=self.transform,
                )
                # Fixed seed makes the split deterministic across all processes.
                gen = torch.Generator().manual_seed(42)
                self.train_set, self.val_set = random_split(
                    full_dataset,
                    [45000, 5000],  # 90% train, 10% validation
                    generator=gen,
                )

        if stage in ("test", "predict", None):
            if not hasattr(self, "test_set"):
                self.test_set = datasets.CIFAR100(
                    self.hparams.data_dir,
                    train=False,
                    transform=self.transform,
                )

    # Step 4: Expose Stage-Specific Loaders
    # A shared helper _make_loader() owns batching policy (batch_size, num_workers, pinning).
    # Each public loader (train_, val_, test_) specifies only its dataset and shuffle policy.
    def _make_loader(self, dataset, shuffle):
        return DataLoader(
            dataset,
            batch_size=self.hparams.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.num_workers > 0,  # Reuse worker processes across epochs
        )

    def train_dataloader(self):
        return self._make_loader(self.train_set, shuffle=True)

    def val_dataloader(self):
        return self._make_loader(self.val_set, shuffle=False)

    def test_dataloader(self):
        return self._make_loader(self.test_set, shuffle=False)

    def predict_dataloader(self):
        return self._make_loader(self.test_set, shuffle=False)


### Step 5: Verify the DataModule Output

In [2]:
data_module = CIFARDataModule(batch_size=64)
data_module.prepare_data()
data_module.setup(stage="fit")

sample_images, sample_labels = next(
    iter(data_module.train_dataloader())
)

assert len(data_module.train_set) == 45000
assert len(data_module.val_set) == 5000
assert sample_images.shape == (64, 3, 32, 32)
assert sample_labels.shape == (64,)

100%|██████████| 169M/169M [26:10<00:00, 108kB/s]  


### Step 6 to 9: Build the Encoder and Decoder

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Project A refactors the native denoising autoencoder from Chapter 8 into a reusable Lightning module.
# The encoder reduces spatial resolution (32×32 → 8×8) while increasing channel depth.
# The decoder reverses this with transposed convolutions to restore the original image shape.
class LitDenoisingAutoencoder(pl.LightningModule):
    # Step 6: Build the Encoder and Decoder
    # The encoder learns compressed representations by downsampling and the decoder
    # reconstructs pixel values. MSE loss drives the reconstruction accuracy.
    def __init__(
        self,
        noise_factor: float = 0.2,  # Corruption strength during training
        lr: float = 2e-3,
    ):
        super().__init__()
        self.save_hyperparameters()

        # Encoder: 3 channels → 32 → 64 channels, spatial resolution 32×32 → 8×8
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        # Decoder: Restores 64 channels back to 3 and expands spatial resolution 8×8 → 32×32
        # Sigmoid ensures output remains in [0, 1] (valid pixel range for normalized images).
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32, 3,
                stride=2, padding=1, output_padding=1,
            ),
            nn.ReLU(),
            nn.ConvTranspose2d(
                32, 3, 3,
                stride=2, padding=1, output_padding=1,
            ),
            nn.Sigmoid(),  # Constrain output to [0, 1]
        )

    # Step 7: Define Reconstruction and Dynamic Corruption
    # Forward pass is deterministic: pass input through encoder and decoder.
    # Corruption (add_noise) is a separate method so it can be controlled or disabled.
    def forward(self, x):
        return self.decoder(self.encoder(x))

    # Add Gaussian noise with controlled intensity, clamp to valid pixel range.
    def add_noise(self, clean_images):
        noise = torch.randn_like(
            clean_images
        ) * self.hparams.noise_factor
        # Clamp ensures noisy images remain in [0, 1] like clean images.
        return torch.clamp(
            clean_images + noise, 0.0, 1.0
        )
    
    # Step 8: Share the Denoising Objective Across Stages
    # _shared_step() contains the common logic: corrupt, reconstruct, compute MSE.
    # Training, validation, and test hooks delegate to this method with different metric prefixes.
    def _shared_step(self, batch, prefix):
        clean_images, _ = batch
        noisy_images = self.add_noise(clean_images)  # Corrupt the clean batch
        reconstructions = self(noisy_images)          # Reconstruct from noisy input
        loss = F.mse_loss(reconstructions, clean_images)  # Compare against clean target

        self.log(
            f"{prefix}_loss",
            loss,
            on_epoch=True,
            prog_bar=True,
            sync_dist=prefix != "train",              # Sync validation/test metrics across processes
            batch_size=clean_images.size(0),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        clean_images, _ = batch
        noisy_images = self.add_noise(clean_images)
        return {
            "noisy": noisy_images,
            "reconstruction": self(noisy_images),
        }
    
    # Step 9: Configure Optimization
    # Adam optimizer with a learning rate that can be adjusted per experiment.
    def configure_optimizers(self):
        return torch.optim.Adam(
            self.parameters(),
            lr=self.hparams.lr,
        )


### Step 9-2: Verify Shapes

In [ ]:
# Before involving the Trainer, verify that the model preserves input shape,
# respects the pixel range [0, 1], and forward pass works in inference mode.
dae_model = LitDenoisingAutoencoder()

with torch.inference_mode():
    # Test on a small batch (first 4 images) to catch shape mismatches early.
    noisy_images = dae_model.add_noise(sample_images[:4])
    reconstructions = dae_model(noisy_images)

# Output must preserve the input shape (batch_size, 3, 32, 32).
assert reconstructions.shape == sample_images[:4].shape
# Sigmoid ensures pixel values stay in [0, 1]; verify this constraint.
assert reconstructions.min().item() >= 0.0
assert reconstructions.max().item() <= 1.0


### Step 10: Verify Trainer Integration

In [5]:
# One-batch smoke run: confirms DataModule setup, optimizer construction, backward pass,
# and Trainer integration all work together. This catches integration bugs before long training.
dae_smoke_trainer = pl.Trainer(
    fast_dev_run=True,       # Run exactly 1 batch through train, val, and test
    accelerator="auto",      # Automatically detect CPU/GPU
    devices=1,               # Use 1 device (no distribution)
    logger=False,            # Disable logging for quick test
    enable_checkpointing=False,  # Disable checkpointing for this quick test
)

dae_smoke_trainer.fit(
    dae_model,
    datamodule=data_module,
)

# After one complete gradient step, confirm the loop executed successfully.
assert dae_smoke_trainer.global_step == 1


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.
You are using a CUDA device ('NVIDIA GeForce RTX 3060') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params | Mode 
-----------------------------------------------
0 | encoder | Sequential | 19.4 K | train
1 | decoder | Sequential | 19.3 K | train
-----------------------------------------------
38.7 K    Trainable params
0         Non-trainable params
38.7 K    Total params
0.155     Total estimated model params size (MB)
10   

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.


## Project B: The Variational Autoencoder

### Step 1 to 3: Building the VAE Core

In [ ]:
# VAE learns a regularized latent distribution instead of a point representation.
# The Encoder outputs distribution parameters (μ, log σ²), not a single vector.
# The Reparameterization Trick makes sampling differentiable so gradients flow back to the encoder.
class VAECore(nn.Module):
    # Step 1: Encode Images into Latent Parameters
    # Unlike a standard autoencoder, the encoder predicts two quantities for each image:
    # - mu: mean of the approximate posterior distribution
    # - logvar: log-variance (log(σ²)), used to parameterize uncertainty
    def __init__(self, latent_dim: int = 128):
        super().__init__()
        self.latent_dim = latent_dim

        # Encoder: Extract spatial features and reduce to a 1D representation.
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),  # Flatten to (batch, 64*8*8=4096)
        )
        # Two independent heads predict distribution parameters.
        self.fc_mu = nn.Linear(64 * 8 * 8, latent_dim)
        self.fc_logvar = nn.Linear(64 * 8 * 8, latent_dim)

        # Decoder input: Project latent vector back to spatial feature map (64, 8, 8).
        self.decoder_input = nn.Linear(
            latent_dim, 64 * 8 * 8
        )
        # Decoder: Transpose convolutions restore spatial resolution and color channels.
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(
                64, 32, 3,
                stride=2, padding=1, output_padding=1,
            ),
            nn.ReLU(),
            nn.ConvTranspose2d(
                32, 3, 3,
                stride=2, padding=1, output_padding=1,
            ),
            nn.Sigmoid(),  # Constrain output to [0, 1]
        )

    # Step 2: Make Sampling Differentiable through the Reparameterization Trick
    # Instead of sampling z ~ N(μ, σ²) directly, write z = μ + σ ⊙ ε
    # where ε ~ N(0, I) is sampled independently. This makes sampling
    # differentiable by converting randomness to a deterministic transformation.
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)  # Convert log(σ²) to σ
        eps = torch.randn_like(std)    # Sample ε ~ N(0, I)
        return mu + eps * std          # z = μ + σ ⊙ ε (differentiable)

    # Step 3: Decode Latent Samples into Images
    def decode(self, z):
        # Project latent vector to spatial feature dimensions (batch, 64, 8, 8).
        z_spatial = self.decoder_input(z).view(
            -1, 64, 8, 8
        )
        # Restore original resolution with transposed convolutions.
        return self.decoder(z_spatial)

    # Step 4: Combine Encoding, Sampling, and Decoding into a Single Forward Pass
    # The forward pass is deterministic but internally stochastic due to reparameterization.
    def forward(self, x):
        h = self.encoder(x)                      # Encode to features
        mu = self.fc_mu(h)                       # Predict mean
        logvar = self.fc_logvar(h)               # Predict log-variance
        z = self.reparameterize(mu, logvar)     # Sample latent code (differentiable)
        return self.decode(z), mu, logvar       # Decode and return distribution parameters


In [ ]:
# Verify output shapes before adding loss or Trainer.
# Each image produces reconstruction, mean vector, and log-variance vector.
vae_core = VAECore(latent_dim=32)

with torch.inference_mode():
    x_hat, mu, logvar = vae_core(sample_images[:4])

# Reconstruction must preserve input shape (batch_size, 3, 32, 32).
assert x_hat.shape == sample_images[:4].shape
# Distribution parameters are (batch_size, latent_dim).
assert mu.shape == (4, 32)
assert logvar.shape == (4, 32)


### Step 4 to 7: Wrap the Mathematical Core with a Lightning Module

In [ ]:
# Lightning wrapper for VAE: owns the loss function, Trainer hooks, and sampling schedule.
# The VAECore remains a pure nn.Module focused on encoding, sampling, and decoding.
class LitVAE(pl.LightningModule):
    def __init__(
        self,
        latent_dim: int = 128,
        lr: float = 1e-3,
        kl_weight: float = 1.0,  # Weight for KL divergence (1.0 is standard VAE; >1.0 is beta-VAE)
    ):
        super().__init__()
        self.save_hyperparameters()
        self.core = VAECore(latent_dim=latent_dim)

    def forward(self, x):
        return self.core(x)
    
    # Step 5: Build the VAE Loss One Term at a Time
    # VAE objective combines two terms:
    # 1. Reconstruction loss: MSE between input and reconstruction (encourages detail)
    # 2. KL divergence: Regularizes latent distribution toward unit Gaussian (enables generation)
    # Minimizing both simultaneously learns both accurate reconstruction and smooth latent space.
    def _shared_step(self, batch, prefix: str):
        x, _ = batch
        x_hat, mu, logvar = self(x)

        # Reconstruction loss: MSE summed over batch, then normalized by batch size.
        recon_loss = F.mse_loss(
            x_hat, x, reduction="sum"
        ) / x.size(0)

        # KL divergence: -0.5 * sum(1 + log(σ²) - μ² - σ²)
        # Measures how far the posterior deviates from N(0, I).
        # Divided by batch size for numerical stability.
        kl_loss = -0.5 * torch.sum(
            1 + logvar - mu.pow(2) - logvar.exp()
        ) / x.size(0)

        # Weighted combination: kl_weight=1.0 is standard VAE.
        # kl_weight > 1 is beta-VAE, emphasizing latent regularization.
        total_loss = recon_loss + (
            self.hparams.kl_weight * kl_loss
        )

        # Log all three terms to monitor reconstruction vs. regularization trade-off.
        self.log_dict(
            {
                f"{prefix}_loss": total_loss,
                f"{prefix}_recon": recon_loss,
                f"{prefix}_kl": kl_loss,
            },
            on_epoch=True,
            prog_bar=prefix != "test",
            sync_dist=(prefix != "train"),  # Sync validation/test metrics across processes
            batch_size=x.size(0),
        )
        return total_loss
    
    # Step 6: Connect the Objective to Managed Stages
    # Public hooks delegate to _shared_step with appropriate metric prefixes.
    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    # Prediction returns reconstructions for use in downstream tasks.
    def predict_step(self, batch, batch_idx):
        x, _ = batch
        x_hat, _, _ = self(x)
        return x_hat
    
    # Step 7: Add Generative Sampling and Optimization
    # Generation (vs. reconstruction) starts from random latent codes, not from encoded images.
    # Once per epoch (when validation completes), sample from the prior and decode to visualize generation quality.
    def on_validation_epoch_end(self):
        # Only the main process logs; avoid duplicate entries in distributed runs.
        if not self.trainer.is_global_zero:
            return

        # Sample latent codes from the standard normal prior N(0, I).
        z = torch.randn(
            16,
            self.hparams.latent_dim,
            device=self.device,
        )
        # Decode random latent codes to synthesize new images.
        synthetic_images = self.core.decode(z)
        
        # Log synthetic images to TensorBoard/W&B for qualitative inspection.
        # This monitors whether the model learns a meaningful latent space and smooth generation.
        self.logger.experiment.add_images("synthetic_images", synthetic_images, self.current_epoch)

    # Adam optimizer with configurable learning rate.
    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
        )


### Step 7-2: Verify the VAE Training Loop

In [ ]:
# Verify that the Lightning wrapper preserves the core's output shapes and behavior.
# This allows _shared_step to correctly compute the VAE loss.
vae_model = LitVAE(latent_dim=32)

with torch.inference_mode():
    x_hat, mu, logvar = vae_model(sample_images[:4])

# Output shapes must match what _shared_step expects.
assert x_hat.shape == sample_images[:4].shape
assert mu.shape == (4, 32)
assert logvar.shape == (4, 32)


### Step 8: Granular Compilation & Execution

In [16]:
# Final capstone: assemble data, model, compilation, precision, and Trainer for end-to-end execution.
# Following the "granular" pattern: each component is tested independently before combining.
import torch

data_module = CIFARDataModule(batch_size=256)
model = LitVAE(latent_dim=128)

use_cuda = torch.cuda.is_available()

# torch.compile() optimizes the model graph (tensor-heavy VAECore) for faster execution.
# It requires careful benchmarking and should only be enabled after verifying correctness.
# This flag allows easy on/off toggling for experiments.
use_compile = False
if use_compile:
    model.core = torch.compile(model.core)  # Compile only the mathematical core, not the Lightning wrapper

# Select numeric precision based on available hardware.
# BF16 on supported CUDA avoids loss scaling and provides good numerical stability.
# FP16 as fallback on older CUDA; FP32 on CPU or unsupported platforms.
if use_cuda and torch.cuda.is_bf16_supported():
    precision = "bf16-mixed"    # Preferred: FP32-like range with lower memory
elif use_cuda:
    precision = "16-mixed"       # Fallback: FP16 with narrower range
else:
    precision = "32-true"        # Safe: Full float32 everywhere

# Configure Trainer with discovered hardware and precision settings.
# max_epochs=30 is a reasonable starting point; early_stop callbacks can terminate earlier.
trainer = pl.Trainer(
    accelerator="auto",           # Auto-detect GPU/CPU/MPS
    devices="auto",               # Auto-detect device count
    precision=precision,          # Apply selected numeric precision
    max_epochs=30,                # Maximum training epochs
)

# Launch training: the Trainer manages all loops, device placement, validation, and callbacks.
trainer.fit(model, datamodule=data_module)


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name | Type    | Params | Mode 
-----------------------------------------
0 | core | VAECore | 1.6 M  | train
-----------------------------------------
1.6 M     Trainable params
0         Non-trainable params
1.6 M     Total params
6.464     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.
